### Configuración de Checkpoint Merge
Define los parámetros para descargar los modelos desde Civitai o enlaces externos, y configura las opciones de fusión (Merge Mode).

In [2]:
# @title Parámetros de Fusión y Descarga

# @markdown ### Autenticación
civitai_api_key = "82ae4bb33ab08cb02f371b9d01cde4c0"  # @param {type:"string"}

# @markdown ### Enlaces de Modelos (Checkpoints)
checkpoint_link_1 = "https://civitai.com/api/download/models/1154?fileId=879"  # @param {type:"string"}
checkpoint_link_2 = "https://civitai.com/api/download/models/50?fileId=483"  # @param {type:"string"}
checkpoint_link_3 = ""  # @param {type:"string"}

# @markdown ### Configuración del Merge
base_model = "SD 1.5"  # @param ["SD 1.5", "SDXL"]
merge_mode = "Weighted Sum"  # @param ["Weighted Sum", "Add Difference", "Sigmoid", "Spline"]
calculation_mode = "fp16"  # @param ["fp16", "fp32", "bf16", "fp8"]
fp16_conversion = True  # @param {type:"boolean"}
alpha = 0.4  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

import os
import requests

def download_model(url, target_path, api_key=None):
    if not url:
        print("Enlace vacío, omitiendo...")
        return None

    # Si es un enlace de Civitai, añadir el token de API si existe
    if "civitai.com" in url and api_key:
        connector = "&" if "?" in url else "?"
        url = f"{url}{connector}token={api_key}"

    print(f"Descargando modelo desde: {url}")
    os.makedirs(os.path.dirname(target_path), exist_ok=True)

    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(target_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Guardado en: {target_path}")
        return target_path
    else:
        print(f"Error al descargar: Código de estado {response.status_code}")
        return None

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


In [15]:
# @title 1. Instalar dependencias necesarias
!pip install -q --upgrade safetensors torch pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 589.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 709.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9

In [21]:
# @title 2. Descargar Checkpoints y Ejecutar Fusión (Merge)
import os
import torch
import io
import pickle
from safetensors.torch import load_file, save_file

# Definir nombres locales para almacenar los archivos descargados
model_path_1 = "/content/model_1.safetensors"
model_path_2 = "/content/model_2.safetensors"
model_path_3 = "/content/model_3.safetensors"
output_path = "/content/merged_model.safetensors"

# 1. Descargar modelos usando las variables definidas en la configuración
print("--- Iniciando descargas de checkpoints ---")
path_1 = download_model(checkpoint_link_1, model_path_1, civitai_api_key) if checkpoint_link_1 else None
path_2 = download_model(checkpoint_link_2, model_path_2, civitai_api_key) if checkpoint_link_2 else None
path_3 = download_model(checkpoint_link_3, model_path_3, civitai_api_key) if checkpoint_link_3 else None

def safe_torch_load(filepath):
    """Carga un checkpoint de PyTorch interceptando clases de Lightning/Pydantic de manera segura mediante monkeypatch."""
    # Guardar referencia original del método de deserialización estándar
    original_find_class = pickle.Unpickler.find_class

    def patched_find_class(self, module, name):
        # Si intenta buscar clases conflictivas de pytorch_lightning o pydantic, devolver una clase Dummy vacía
        if any(x in module for x in ["pytorch_lightning", "lightning", "pydantic"]):
            class Dummy:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return Dummy
        try:
            return original_find_class(self, module, name)
        except Exception:
            class DummyFallback:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return DummyFallback

    # Aplicar el parche temporal en el módulo pickle importado globalmente
    pickle.Unpickler.find_class = patched_find_class

    try:
        print("-> Cargando archivo con deserializador seguro...")
        with open(filepath, 'rb') as f:
            buffer = io.BytesIO(f.read())
        # Se carga usando el módulo pickle estándar (parcheado temporalmente)
        checkpoint = torch.load(buffer, map_location='cpu')
    finally:
        # Garantizar que el método original sea restaurado siempre
        pickle.Unpickler.find_class = original_find_class

    return checkpoint

def load_model_weights(filepath):
    """Carga pesos de forma segura detectando si es PyTorch o Safetensors."""
    if not filepath or not os.path.exists(filepath):
        return None

    # Comprobar la firma del archivo (PK\x03\x04 indica un archivo zip/pytorch ckpt)
    with open(filepath, 'rb') as f:
        header = f.read(4)

    if header == b'PK\x03\x04':
        print(f"-> Detectado formato PyTorch (.ckpt) para {filepath}.")
        try:
            checkpoint = safe_torch_load(filepath)
        except Exception as e:
            print(f"Fallo crítico en la carga del archivo PyTorch: {e}")
            raise e

        # Los checkpoints de SD suelen tener los pesos bajo 'state_dict'
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            return checkpoint['state_dict']
        return checkpoint
    else:
        print(f"-> Detectado formato Safetensors para {filepath}. Cargando con load_file...")
        return load_file(filepath)

def run_safetensors_merge():
    if not path_1 or not path_2:
        print("Error: Se requieren al menos el Checkpoint 1 y 2 para fusionar.")
        return

    print(f"\n--- Leyendo tensores de los archivos para modelo base {base_model} ---")
    dict_a = load_model_weights(path_1)
    dict_b = load_model_weights(path_2)
    dict_c = load_model_weights(path_3) if (merge_mode == "Add Difference" and path_3) else None

    # Determinar tipo de dato de cálculo
    if calculation_mode == "fp16":
        dtype = torch.float16
    elif calculation_mode == "bf16":
        dtype = torch.bfloat16
    elif calculation_mode == "fp8":
        dtype = torch.float8_e4m3fn if hasattr(torch, 'float8_e4m3fn') else torch.float16
        if dtype == torch.float16:
            print("Advertencia: FP8 no está soportado en este entorno de PyTorch, usando FP16 como respaldo.")
    else:
        dtype = torch.float32

    merged_dict = {}
    print(f"Aplicando fusión: Mode = {merge_mode}, Alpha = {alpha}, Precision = {calculation_mode}")

    keys_a = set(dict_a.keys())
    keys_b = set(dict_b.keys())
    common_keys = keys_a.intersection(keys_b)

    print(f"Total de llaves compartidas encontradas: {len(common_keys)}")

    for key in common_keys:
        t_a = dict_a[key]
        t_b = dict_b[key]

        # Asegurarse de que ambos sean tensores de PyTorch antes de convertir
        if not isinstance(t_a, torch.Tensor) or not isinstance(t_b, torch.Tensor):
            continue

        t_a = t_a.to(dtype)
        t_b = t_b.to(dtype)

        if merge_mode == "Weighted Sum":
            merged_tensor = (1.0 - alpha) * t_a + alpha * t_b
        elif merge_mode == "Add Difference" and dict_c is not None and key in dict_c:
            t_c = dict_c[key]
            if isinstance(t_c, torch.Tensor):
                t_c = t_c.to(dtype)
                merged_tensor = t_a + alpha * (t_b - t_c)
            else:
                merged_tensor = t_a
        else:
            # Si no aplica otro modo, por defecto se asigna el tensor del modelo A
            merged_tensor = t_a

        # Forzar guardado en fp16 si fp16_conversion está activo
        if fp16_conversion:
            merged_tensor = merged_tensor.to(torch.float16)

        merged_dict[key] = merged_tensor

    # Copiar llaves únicas del modelo A que no estuvieran en B
    for key in keys_a - common_keys:
        t_a = dict_a[key]
        if isinstance(t_a, torch.Tensor):
            merged_dict[key] = t_a.to(torch.float16) if fp16_conversion else t_a

    print(f"Guardando resultado final en: {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_file(merged_dict, output_path)
    print("¡El proceso de fusión ha finalizado de manera exitosa!")

try:
    run_safetensors_merge()
except Exception as e:
    print(f"Ocurrió un error inesperado durante el merge: {e}")

--- Iniciando descargas de checkpoints ---
Descargando modelo desde: https://civitai.com/api/download/models/1154?fileId=879&token=82ae4bb33ab08cb02f371b9d01cde4c0
Guardado en: /content/model_1.safetensors
Descargando modelo desde: https://civitai.com/api/download/models/50?fileId=483&token=82ae4bb33ab08cb02f371b9d01cde4c0
Guardado en: /content/model_2.safetensors

--- Leyendo tensores de los archivos para modelo base SD 1.5 ---
-> Detectado formato PyTorch (.ckpt) para /content/model_1.safetensors.
Fallo crítico en la carga del archivo PyTorch: cannot set 'find_class' attribute of immutable type '_pickle.Unpickler'
Ocurrió un error inesperado durante el merge: cannot set 'find_class' attribute of immutable type '_pickle.Unpickler'


### Guardar Modelo en Google Drive
Usa las siguientes celdas para montar tu Google Drive y transferir el modelo resultante para guardarlo de manera permanente.

In [ ]:
# @title 3. Conectar a Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title 4. Copiar Modelo Fusionado a Google Drive
import shutil
import os

# @markdown Ruta de la carpeta de destino en tu Google Drive
drive_folder = "/content/drive/MyDrive/StableDiffusion/Models" # @param {type:"string"}

os.makedirs(drive_folder, exist_ok=True)

source_file = "/content/merged_model.safetensors"
target_file = os.path.join(drive_folder, "merged_model_xl.safetensors")

if os.path.exists(source_file):
    print(f"Copiando {source_file} a {target_file}...")
    shutil.copy(source_file, target_file)
    print("¡Modelo copiado con éxito a tu Google Drive!")
else:
    print("Error: El archivo de modelo fusionado no existe en /content/merged_model.safetensors. Asegúrate de ejecutar la celda de fusión primero.")

Copiando /content/merged_model.safetensors a /content/drive/MyDrive/StableDiffusion/Models/merged_model_xl.safetensors...
¡Modelo copiado con éxito a tu Google Drive!
